# Chunking Strategies for RAG

A hands-on tour of the chunking strategies commonly used when building retrieval pipelines. Each section has:

1. **What it is / when to use it**
2. **`pip install`** for the library it needs
3. **Runnable code** against a shared sample document

Strategies covered:

| # | Strategy | Library |
|---|----------|---------|
| 1 | Fixed-size character chunking | `langchain-text-splitters` |
| 2 | Recursive character chunking | `langchain-text-splitters` |
| 3 | Token-based chunking | `tiktoken` |
| 4 | Sentence-based chunking | `nltk` |
| 5 | Sliding window with overlap | pure Python |
| 6 | Markdown/header-aware chunking | `langchain-text-splitters` |
| 7 | Semantic (embedding similarity) chunking | `langchain-experimental` + `sentence-transformers` |
| 8 | Document-layout aware chunking (title/section detection) | `unstructured` |
| 9 | LlamaIndex node parsers | `llama-index-core` |
| 10 | Agentic / LLM-based chunking | any LLM client (conceptual + code) |
| 11 | Cluster-based chunking | `scikit-learn` + `sentence-transformers` |
| 12 | Document-aware (type/language) chunking | `langchain-text-splitters` |
| 13 | Metadata filtering (retrieval-time, not chunking) | `sentence-transformers` |


In [3]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 17.3 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 6.1 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Install everything used in this notebook (safe to re-run).
# Comment out sections you don't need — some (sentence-transformers/unstructured) download models/large deps.
# Note: %pip does not support inline "# comment" text after arguments — keep comments on their own line.

%pip install -q langchain-text-splitters tiktoken nltk
# semantic + cluster chunking
%pip install -q langchain-experimental langchain-community sentence-transformers scikit-learn
# layout-aware chunking
%pip install -q "unstructured[md]"
# LlamaIndex node parsers
%pip install -q llama-index-core llama-index-embeddings-huggingface


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Sample document

A synthetic Markdown doc with headers, paragraphs of varying length, a list, and a code block — enough structure to show the difference between strategies.

In [2]:
sample_text = """# Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

## Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

A good chunking strategy tries to balance three things:

- **Semantic coherence** — each chunk should represent one idea or topic.
- **Context preservation** — enough surrounding text should be kept that the chunk is understandable on its own.
- **Retrieval granularity** — chunks should be small enough that a similarity search can pinpoint the relevant passage without pulling in unrelated text.

## Naive Fixed-Size Splitting

The simplest approach slices text every N characters or tokens, regardless of sentence or paragraph boundaries.

```python
def naive_chunks(text, size=200):
    return [text[i:i+size] for i in range(0, len(text), size)]
```

This is fast and library-free, but it can cut sentences (and even words) in half, which hurts embedding quality.

## Structure-Aware Splitting

Better splitters respect the document's natural structure — paragraphs, sentences, Markdown headers, or HTML tags — before falling back to a hard character limit. Recursive splitters try a list of separators in order (for example `["\\n\\n", "\\n", ". ", " "]`) and only fall back to a harder cut when a piece still exceeds the target size.

## Semantic Chunking

Semantic chunkers go one step further: they embed individual sentences, measure the similarity between consecutive sentences, and cut a new chunk wherever the topic shifts significantly. This tends to produce chunks that align with actual topic boundaries rather than arbitrary length limits, at the cost of needing an embedding model at chunking time.

## Choosing a Strategy

For most production RAG systems, a recursive character or token splitter with a modest overlap (10-20% of chunk size) is a strong default. Semantic and layout-aware chunkers are worth the extra cost when documents have highly variable structure (contracts, manuals, mixed HTML/PDF sources) or when retrieval quality on those documents is measurably poor with the simpler approach.
"""

print(sample_text[:300], "...")
print(f"\ntotal characters: {len(sample_text)}")

# Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time ...

total characters: 2599


In [3]:
# Also persist it to disk — the `unstructured` strategy below partitions from a file.
from pathlib import Path

sample_path = Path("sample_doc.md")
sample_path.write_text(sample_text, encoding="utf-8")
sample_path

WindowsPath('sample_doc.md')

## 1. Fixed-Size Character Chunking

**What it is:** split the text every `chunk_size` characters, optionally on a single separator, with a fixed `chunk_overlap` carried into the next chunk.

**When to use it:** quick prototypes, or text with no real structure (logs, raw transcripts). It's the fastest and dumbest option — it will happily cut a sentence in half.

```
pip install langchain-text-splitters
```

In [4]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="\n\n",      # try to break on paragraph boundaries first
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
)

fixed_chunks = fixed_splitter.split_text(sample_text)

print(f"{len(fixed_chunks)} chunks\n")
for i, c in enumerate(fixed_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

c:\Users\amanm\Downloads\Machine_Learning\production-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Created a chunk of size 350, which is longer than the specified 300
Created a chunk of size 340, which is longer than the specified 300
Created a chunk of size 337, which is longer than the specified 300
Created a chunk of size 352, which is longer than the specified 300


12 chunks

--- chunk 0 (32 chars) ---
# Retrieval-Augmented Generation

--- chunk 1 (292 chars) ---
Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

## Why Chunking Matters

--- chunk 2 (350 chars) ---
Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

--- chunk 3 (55 chars) ---
A good chunking strategy tries to balance three things:

--- chunk 4 (340 chars) ---
- **Semantic coherence** — each chunk should represent one idea or topic.
- **Context preservation** — enough surr

## 2. Recursive Character Chunking

**What it is:** tries a prioritized list of separators (`["\n\n", "\n", ". ", " ", ""]` by default) — it splits on the first separator, and only recurses into a finer separator for pieces that are still too big. This keeps paragraphs/sentences intact far more often than a single fixed separator.

**When to use it:** the default choice for most plain-text or lightly structured documents. This is what `RecursiveCharacterTextSplitter` implements and it's the most widely used splitter in production RAG systems.

```
pip install langchain-text-splitters
```

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_chunks = recursive_splitter.split_text(sample_text)

print(f"{len(recursive_chunks)} chunks\n")
for i, c in enumerate(recursive_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

17 chunks

--- chunk 0 (32 chars) ---
# Retrieval-Augmented Generation

--- chunk 1 (292 chars) ---
Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

## Why Chunking Matters

--- chunk 2 (108 chars) ---
Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them

--- chunk 3 (242 chars) ---
. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

--- chunk 4 (55 chars) ---
A good chunking strategy tries to balance three things:

--- chunk 5 (186 chars) ---
- **Semantic coherence** — each chunk should represent one idea or topic.
- **Contex

## 3. Token-Based Chunking

**What it is:** splits by token count (using the same tokenizer as the target LLM/embedding model) instead of raw characters. This matters because embedding models and LLM context windows are limited in *tokens*, not characters, and languages/tokenizers vary in characters-per-token.

**When to use it:** whenever you need to guarantee a chunk fits under a hard token budget (embedding model max input, LLM context window).

```
pip install tiktoken langchain-text-splitters
```

In [6]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",   # tokenizer used by e.g. text-embedding-3 / gpt-4
    chunk_size=60,                 # in tokens, not characters
    chunk_overlap=10,
)

token_chunks = token_splitter.split_text(sample_text)

enc = tiktoken.get_encoding("cl100k_base")
print(f"{len(token_chunks)} chunks\n")
for i, c in enumerate(token_chunks):
    n_tokens = len(enc.encode(c))
    print(f"--- chunk {i} ({n_tokens} tokens) ---\n{c}\n")

18 chunks

--- chunk 0 (59 tokens) ---
# Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

--- chunk 1 (5 tokens) ---
## Why Chunking Matters

--- chunk 2 (59 tokens) ---
Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make

--- chunk 3 (14 tokens) ---
small lose the surrounding context a reader needs to make sense of them.

--- chunk 4 (11 tokens) ---
A good chunking strategy tries to balance three things:

--- chunk 5 (36 tokens) ---
- **Semantic coherence** —

## 4. Sentence-Based Chunking

**What it is:** split into sentences with a proper sentence tokenizer (not just `.split(". ")`, which breaks on abbreviations like "e.g." or "Dr."), then greedily pack sentences into chunks up to a size limit.

**When to use it:** when sentence boundaries matter more than exact chunk size — e.g. legal, medical, or narrative text where mid-sentence cuts are especially damaging.

```
pip install nltk
```

In [7]:
import nltk

nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

def sentence_chunks(text: str, max_chars: int = 300) -> list[str]:
    sentences = sent_tokenize(text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current = f"{current} {sent}".strip()
        else:
            if current:
                chunks.append(current)
            current = sent
    if current:
        chunks.append(current)
    return chunks

# strip markdown syntax first — sentence tokenizers assume prose, not markup
import re
prose_only = re.sub(r"[#*`]|```python.*?```", "", sample_text, flags=re.S)

sent_chunks = sentence_chunks(prose_only, max_chars=300)
print(f"{len(sent_chunks)} chunks\n")
for i, c in enumerate(sent_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

11 chunks

--- chunk 0 (299 chars) ---
Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

--- chunk 1 (131 chars) ---
Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them.

--- chunk 2 (240 chars) ---
The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

--- chunk 3 (235 chars) ---
A good chunking strategy tries to balance three things:

- Semantic coherence — each chunk should represent one idea or topic. - Context preservation — enough surrounding text should be kept that the c

## 5. Sliding Window with Overlap

**What it is:** a fixed-size window slides over the text with a configurable stride < window size, so consecutive chunks overlap. This is the mechanism *inside* most of the splitters above (`chunk_overlap`), shown here standalone so the idea is explicit.

**When to use it:** when you want explicit control over overlap ratio without pulling in a library, or to overlap already-produced chunks (e.g. sentence or paragraph chunks) rather than raw characters.

No install needed — pure Python.

In [8]:
def sliding_window_chunks(text: str, window: int = 300, overlap: int = 100) -> list[str]:
    if overlap >= window:
        raise ValueError("overlap must be smaller than window")
    stride = window - overlap
    return [text[i:i + window] for i in range(0, len(text), stride) if text[i:i + window].strip()]

window_chunks = sliding_window_chunks(sample_text, window=300, overlap=100)
print(f"{len(window_chunks)} chunks\n")
for i, c in enumerate(window_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

13 chunks

--- chunk 0 (300 chars) ---
# Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time

--- chunk 1 (300 chars) ---
stead of relying purely on parametric memory, the model is given relevant passages at inference time.

## Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval q

--- chunk 2 (300 chars) ---
s before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

A good chunking stra


## 6. Markdown / Header-Aware Chunking

**What it is:** splits on structural elements (`#`, `##`, ... headers, or HTML tags) first, attaching each section's header path as metadata, then optionally runs a character/token splitter within each section for further size control.

**When to use it:** Markdown docs, wikis, README-style content, or any HTML/structured doc where headers define meaningful topic boundaries — you get chunks that carry their section context (e.g. `{"Header 1": "RAG", "Header 2": "Why Chunking Matters"}`) as metadata for citations or filtering.

```
pip install langchain-text-splitters
```

In [9]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_docs = md_splitter.split_text(sample_text)

print(f"{len(md_docs)} sections\n")
for i, doc in enumerate(md_docs):
    print(f"--- section {i} | metadata={doc.metadata} ---\n{doc.page_content[:200]}...\n")

6 sections

--- section 0 | metadata={'Header 1': 'Retrieval-Augmented Generation'} ---
Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on paramet...

--- section 1 | metadata={'Header 1': 'Retrieval-Augmented Generation', 'Header 2': 'Why Chunking Matters'} ---
Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too ...

--- section 2 | metadata={'Header 1': 'Retrieval-Augmented Generation', 'Header 2': 'Naive Fixed-Size Splitting'} ---
The simplest approach slices text every N characters or tokens, regardless of sentence or paragraph boundaries.  
```python
def naive_chunks(text, size=200):
return [text[i:i+size] for i in range(0, l...

--- section 3 | metadata={'Header 1': 'Retrieval-Augmented Generati

## 7. Semantic Chunking (Embedding Similarity)

**What it is:** embed each sentence, walk through consecutive sentences comparing embedding similarity (or distance), and cut a new chunk whenever similarity drops below a threshold (i.e. the topic shifts). Chunk boundaries are determined by meaning, not length.

**When to use it:** documents where topics genuinely shift mid-paragraph and length-based cuts produce chunks mixing two topics. Costs more — every sentence needs an embedding call at chunk time — so it's usually reserved for cases where naive splitting has been shown to hurt retrieval.

```
pip install langchain-experimental sentence-transformers
```

In [10]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

# Small local model — no API key required. Swap for OpenAIEmbeddings() etc. in production.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",  # cut where similarity drop exceeds this percentile
    breakpoint_threshold_amount=80,
)

semantic_chunks = semantic_splitter.split_text(prose_only)

print(f"{len(semantic_chunks)} chunks\n")
for i, c in enumerate(semantic_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

C:\Users\amanm\AppData\Local\Temp\ipykernel_828\1200155067.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
C:\Users\amanm\AppData\Local\Temp\ipykernel_828\1200155067.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\amanm\Downloads\Machine_Learning\production-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symli

4 chunks

--- chunk 0 (300 chars) ---
 Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

--- chunk 1 (1307 chars) ---
Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them. A good chunking strategy tries to balance three things:

- Semantic coherence — each chunk should represent one idea or topic. - Context preservation — enough surrounding text should be kept that the chunk is understandable on its own. - Retrieval granularit

## 8. Document-Layout Aware Chunking (`unstructured`)

**What it is:** `unstructured` parses a real file (PDF, DOCX, HTML, Markdown, PPTX, images via OCR...) into typed elements (`Title`, `NarrativeText`, `Table`, `ListItem`, ...), then `chunk_by_title` groups elements into chunks that each start at a `Title` element and never cross a section boundary. This is layout/structure detection, not just markup parsing — it works on scanned PDFs and DOCX too, not just Markdown.

**When to use it:** messy real-world documents (PDFs with multi-column layouts, scanned reports, DOCX exports) where you need actual document understanding, not just text-based heuristics.

```
pip install "unstructured[md]"
```

In [11]:
from unstructured.partition.md import partition_md
from unstructured.chunking.title import chunk_by_title

elements = partition_md(filename=str(sample_path))
title_chunks = chunk_by_title(elements, max_characters=400, combine_text_under_n_chars=100)

print(f"{len(title_chunks)} chunks\n")
for i, c in enumerate(title_chunks):
    print(f"--- chunk {i} ({len(c.text)} chars) ---\n{c.text}\n")

8 chunks

--- chunk 0 (299 chars) ---
Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

--- chunk 1 (372 chars) ---
Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

--- chunk 2 (381 chars) ---
A good chunking strategy tries to balance three things:

Semantic coherence — each chunk should represent one idea or topic.

Context preservation — enough surrounding text should be kept that the chunk is understandable on its own

## 9. LlamaIndex Node Parsers

**What it is:** LlamaIndex's equivalent of text splitters, but operating on `Document`/`Node` objects that carry metadata and relationships through the pipeline. `SentenceSplitter` is the recursive/sentence-aware default; `SemanticSplitterNodeParser` is LlamaIndex's embedding-based semantic chunker (same idea as #7, different implementation).

**When to use it:** if the rest of your indexing/retrieval stack is already built on LlamaIndex — its node parsers integrate directly with its indices, retrievers, and query engines.

```
pip install llama-index-core llama-index-embeddings-huggingface
```

In [12]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

doc = Document(text=prose_only)

# 9a. Plain sentence-aware splitter (fast, no embeddings needed)
sentence_parser = SentenceSplitter(chunk_size=300, chunk_overlap=50)
sentence_nodes = sentence_parser.get_nodes_from_documents([doc])
print(f"SentenceSplitter -> {len(sentence_nodes)} nodes")
for i, n in enumerate(sentence_nodes):
    print(f"--- node {i} ({len(n.text)} chars) ---\n{n.text}\n")

# 9b. Semantic splitter (embedding-based, mirrors strategy #7)
li_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
semantic_parser = SemanticSplitterNodeParser(embed_model=li_embed_model, breakpoint_percentile_threshold=80)
semantic_nodes = semantic_parser.get_nodes_from_documents([doc])
print(f"\nSemanticSplitterNodeParser -> {len(semantic_nodes)} nodes")
for i, n in enumerate(semantic_nodes):
    print(f"--- node {i} ({len(n.text)} chars) ---\n{n.text}\n")

SentenceSplitter -> 2 nodes
--- node 0 (1308 chars) ---
Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

 Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

A good chunking strategy tries to balance three things:

- Semantic coherence — each chunk should represent one idea or topic.
- Context preservation — enough surrounding text should be kept that the chunk is understandable on its own.
- Retrieval granularity — chunks

c:\Users\amanm\Downloads\Machine_Learning\production-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amanm\AppData\Local\llama_index\llama_index\Cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 


SemanticSplitterNodeParser -> 4 nodes
--- node 0 (303 chars) ---
 Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) combines a retriever with a generator so that a language model can answer questions using information it was never trained on. Instead of relying purely on parametric memory, the model is given relevant passages at inference time.

 

--- node 1 (1314 chars) ---
Why Chunking Matters

Chunking is the process of splitting a large document into smaller pieces before embedding and indexing them. The size and boundaries of a chunk directly affect retrieval quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that are too small lose the surrounding context a reader needs to make sense of them.

A good chunking strategy tries to balance three things:

- Semantic coherence — each chunk should represent one idea or topic.
- Context preservation — enough surrounding text should be kept that the chunk is understandable on 

## 10. Agentic / LLM-Based Chunking

**What it is:** instead of a rule/embedding-based heuristic, an LLM reads the document and directly proposes chunk boundaries (or writes a standalone "proposition" per idea, à la [dense X / propositional retrieval](https://arxiv.org/abs/2312.06648)). The model can reason about where one self-contained idea ends and another begins, merge a short trailing sentence into the previous chunk, name each chunk, etc.

**When to use it:** highest quality, highest cost/latency — reserve it for a one-time offline indexing pass over a small-to-medium high-value corpus (e.g. legal contracts, policy docs) where retrieval quality matters more than indexing cost. Not something you'd run per-query.

No fixed pip package — this is a prompt against whatever LLM client you already use. Shown below with **Vertex AI's Gemini**, the same client used throughout `vertex_rag_pipeline.ipynb` in this folder.

```
pip install google-cloud-aiplatform
```
Requires a `PROJECT_ID` with the Vertex AI API enabled and `gcloud auth application-default login` run beforehand — the cell below is skipped automatically if `PROJECT_ID` isn't set.


In [ ]:
import json

PROJECT_ID = ""  # TODO: set to your GCP project id to run this cell
LOCATION = "us-central1"

CHUNKING_PROMPT = """Split the following document into self-contained chunks for a retrieval system.
Rules:
- Each chunk must be understandable on its own, without needing the other chunks.
- Do not cut a sentence or idea in half.
- Prefer 100-400 words per chunk; merge very short trailing sections into the previous chunk.
Return ONLY a JSON array of strings, one per chunk.

DOCUMENT:
{document}
"""

def llm_chunks(text: str) -> list[str]:
    response = generative_model.generate_content(CHUNKING_PROMPT.format(document=text))
    raw = response.text.strip().strip("`").removeprefix("json").strip()
    return json.loads(raw)

if PROJECT_ID:
    import vertexai
    from vertexai.generative_models import GenerativeModel

    vertexai.init(project=PROJECT_ID, location=LOCATION)
    generative_model = GenerativeModel("gemini-2.0-flash-001")

    agentic_chunks = llm_chunks(prose_only)
    print(f"{len(agentic_chunks)} chunks\n")
    for i, c in enumerate(agentic_chunks):
        print(f"--- chunk {i} ---\n{c}\n")
else:
    print("PROJECT_ID not set — skipping. Set PROJECT_ID above and run `gcloud auth application-default login` first.")


## Additional Sample Assets

Two more strategies need richer inputs than a single short doc:

- **`sample_document.md`** — a 5-section, multi-topic doc (RAG, vector DBs, embeddings, prompting, evaluation). Good for cluster-based chunking (distinct topics to cluster) and metadata filtering (each section becomes a chunk with header metadata).
- **`sample_code.py`** — a small Python module (function + class + docstrings). Good for document-aware / language-aware chunking, which needs to respect code structure rather than prose structure.

Both files were written to `experiment/` alongside this notebook.

In [ ]:
multi_topic_text = Path("sample_document.md").read_text(encoding="utf-8")
code_text = Path("sample_code.py").read_text(encoding="utf-8")

print(f"sample_document.md: {len(multi_topic_text)} chars")
print(f"sample_code.py: {len(code_text)} chars")

## 11. Cluster-Based Chunking

**What it is:** embed every sentence, then run a clustering algorithm (e.g. `AgglomerativeClustering` with a distance threshold, or `KMeans` with a fixed k) over the embeddings instead of a sequential similarity threshold. Consecutive sentences that land in the same cluster are merged into one chunk. This differs from strategy #7 (semantic chunking): semantic chunking only compares *neighboring* sentences one pair at a time, while clustering looks at the *global* structure of the embedding space, so it's more robust when a topic gradually drifts rather than jumping sharply.

**When to use it:** long documents with soft topic drift (research papers, meeting transcripts, blog posts) where a single similarity threshold is hard to tune, and you're fine with clustering's extra compute + the need to pick (or estimate) a number of clusters.

```
pip install scikit-learn sentence-transformers
```

In [ ]:
import re
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

multi_topic_prose = re.sub(r"^#.*$", "", multi_topic_text, flags=re.M)  # drop header lines
sentences = [s for s in sent_tokenize(multi_topic_prose) if s.strip()]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
sentence_vecs = embedder.encode(sentences, normalize_embeddings=True)

# distance_threshold controls how eagerly sentences split into separate clusters —
# lower = more, smaller clusters. n_clusters must be None to use a threshold instead of a fixed k.
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.6,
    metric="cosine",
    linkage="average",
)
labels = clustering.fit_predict(sentence_vecs)

# merge consecutive sentences that share a cluster label into one chunk (keeps chunks contiguous)
cluster_chunks, current_label, current_sentences = [], labels[0], [sentences[0]]
for label, sent in zip(labels[1:], sentences[1:]):
    if label == current_label:
        current_sentences.append(sent)
    else:
        cluster_chunks.append((current_label, " ".join(current_sentences)))
        current_label, current_sentences = label, [sent]
cluster_chunks.append((current_label, " ".join(current_sentences)))

print(f"{len(sentences)} sentences -> {len(cluster_chunks)} cluster-based chunks\n")
for i, (label, c) in enumerate(cluster_chunks):
    print(f"--- chunk {i} | cluster={label} ({len(c)} chars) ---\n{c}\n")

## 12. Document-Aware Chunking

**What it is:** pick the splitter — and its separator hierarchy — based on the *type* of document being chunked, rather than using one generic splitter for everything. Two parts:

1. **Language/format-aware splitting**: `RecursiveCharacterTextSplitter.from_language(...)` ships separator hierarchies tuned per language (Python, JS, Markdown, HTML, LaTeX, ...) — e.g. for Python it prefers to split before `\nclass `, `\ndef `, `\n\tdef ` before falling back to blank lines, so it doesn't sever a function body from its `def` line.
2. **A type dispatcher**: an ingestion pipeline that routes each incoming file to the right strategy by extension/MIME type — code → language splitter, Markdown/HTML → header-aware splitter (#6), PDFs/DOCX → layout-aware (#8), plain text → recursive (#2). This is what "chunking" usually means in a real multi-format ingestion pipeline (like `app/ingestion/processor.py` in this repo) rather than a single notebook example.

**When to use it:** any pipeline that ingests more than one document type — treating a `.py` file and a `.pdf` the same way wastes the structure you already know about the source.

```
pip install langchain-text-splitters
```

In [ ]:
from langchain_text_splitters import Language

# 12a. Language-aware splitting — respects function/class boundaries instead of blank lines
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=200, chunk_overlap=0
)
code_chunks = python_splitter.split_text(code_text)

print(f"{len(code_chunks)} code chunks\n")
for i, c in enumerate(code_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---\n{c}\n")

In [ ]:
# 12b. A minimal "document-aware" dispatcher: pick the strategy by file type.

def chunk_document(path: str, chunk_size: int = 300, chunk_overlap: int = 40) -> list[str]:
    p = Path(path)
    text = p.read_text(encoding="utf-8")

    if p.suffix == ".py":
        splitter = RecursiveCharacterTextSplitter.from_language(
            Language.PYTHON, chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )
        return splitter.split_text(text)

    if p.suffix in (".md", ".markdown"):
        sections = MarkdownHeaderTextSplitter(
            headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
        ).split_text(text)
        sub_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        return [c for sec in sections for c in sub_splitter.split_text(sec.page_content)]

    # fall back to the generic recursive splitter for anything else (.txt, .html, ...)
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap).split_text(text)


for path in ["sample_code.py", "sample_document.md"]:
    result = chunk_document(path)
    print(f"{path}: {len(result)} chunks (strategy chosen by extension)")

## 13. Metadata Filtering

**What it is:** not a chunking strategy by itself — it's what you do *with* the metadata a good chunker attaches. Each chunk carries fields like `source`, `Header 1`/`Header 2` (section), `doc_type`, `date`. At query time, a metadata filter narrows the candidate set *before* (or in addition to) similarity ranking, so a query can't retrieve a technically-similar chunk from the wrong section/source/time range.

Below: split `sample_document.md` with the header splitter (#6), attach extra document-level metadata, embed each chunk, then compare an **unfiltered** search against one **filtered** to a specific section — showing the filter changing which chunk wins even when it isn't the top unfiltered match.

```
pip install sentence-transformers
```

In [ ]:
# Build chunks with metadata (source + section headers), then embed each one.
sections = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
).split_text(multi_topic_text)

for sec in sections:
    sec.metadata["source"] = "sample_document.md"
    sec.metadata["ingested_at"] = "2026-07-19"

texts = [s.page_content for s in sections]
metadatas = [s.metadata for s in sections]
chunk_vecs = embedder.encode(texts, normalize_embeddings=True)

def search(query: str, metadata_filter: dict | None = None, top_k: int = 3):
    candidates = list(range(len(texts)))
    if metadata_filter:
        candidates = [
            i for i in candidates
            if all(metadatas[i].get(k) == v for k, v in metadata_filter.items())
        ]
    if not candidates:
        return []
    query_vec = embedder.encode([query], normalize_embeddings=True)[0]
    scored = sorted(
        ((float(np.dot(query_vec, chunk_vecs[i])), i) for i in candidates),
        reverse=True,
    )
    return [(score, metadatas[i], texts[i][:120]) for score, i in scored[:top_k]]

query = "what should I track to know retrieval quality is good"

print("Unfiltered search:")
for score, meta, snippet in search(query):
    print(f"  {score:.3f} | {meta.get('Header 2', meta.get('Header 1'))} -> {snippet}...")

print("\nFiltered to metadata {'Header 2': 'Vector Databases'}:")
for score, meta, snippet in search(query, metadata_filter={"Header 2": "Vector Databases"}):
    print(f"  {score:.3f} | {meta.get('Header 2')} -> {snippet}...")

## Summary

| Strategy | Boundary respected | Needs embeddings/LLM | Relative cost | Good default for |
|---|---|---|---|---|
| Fixed-size | none | no | lowest | quick prototypes |
| Recursive character | paragraph/sentence | no | low | **general-purpose default** |
| Token-based | paragraph/sentence, token-exact size | no | low | strict context-window budgets |
| Sentence-based | sentence | no | low | narrative/legal/medical text |
| Sliding window | none (explicit overlap) | no | lowest | custom overlap control |
| Markdown/header-aware | document structure | no | low | Markdown/wiki/HTML docs |
| Semantic (embedding) | topic/meaning (local, sequential) | yes (embeddings) | medium | mixed-topic prose |
| `unstructured` layout-aware | document structure (real layout) | no (uses layout model) | medium-high | PDFs, DOCX, scanned docs |
| LlamaIndex node parsers | sentence or topic | optional | low-medium | LlamaIndex-based stacks |
| Agentic/LLM-based | idea/proposition | yes (LLM) | highest | small high-value corpora, offline |
| Cluster-based | topic/meaning (global) | yes (embeddings) | medium-high | long docs with gradual topic drift |
| Document-aware (type/language) | code/format structure | no | low | multi-format ingestion pipelines |
| Metadata filtering | n/a — retrieval-time, not chunk-time | no | low | any corpus with source/section/date fields |

**Rule of thumb:** start with recursive character or token-based chunking + ~15% overlap. Reach for markdown/layout-aware or document-aware (type-routed) splitters when the source has real structure. Reach for semantic or cluster-based chunking only after you've measured that boundary quality — not chunk size — is the retrieval bottleneck; clustering tends to win over sequential semantic chunking on longer documents with slow topic drift. Attach rich metadata to every chunk regardless of strategy — it's what makes metadata filtering possible at query time, and it's far cheaper to add during chunking than to backfill later.